In [1]:
import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go



path_2024 = "../data/raw/snapshots_2026-02-10_ID-2024-hourly.csv"
path_2025 = "../data/raw/snapshots_2026-02-10_ID-2025-hourly.csv"

output_dir = "../data/processed"
output_path = os.path.join(
    output_dir,
    "dataset_CI_RE_2024_2025_cleaned.csv"
)

os.makedirs(output_dir, exist_ok=True)

df_2024 = pd.read_csv(path_2024)
df_2025 = pd.read_csv(path_2025)

df_raw = pd.concat(
    [df_2024, df_2025],
    ignore_index=True
)

print("2024:", df_2024.shape)
print("2025:", df_2025.shape)
print("Combined:", df_raw.shape)

2024: (8784, 11)
2025: (8760, 11)
Combined: (17544, 11)


In [2]:
COL_MAP = {
    "datetime": "datetime",
    "Carbon intensity gCO₂eq/kWh (direct)": "carbon_intensity",
    "Renewable energy percentage (RE%)": "renewable_percentage"
}

df = (
    df_raw[list(COL_MAP.keys())]
    .rename(columns=COL_MAP)
    .copy()
)

df.head()

,datetime,carbon_intensity,renewable_percentage
0,2024-01-01T00:00:00.000000,584.32,16.06
1,2024-01-01T01:00:00.000000,580.70,16.19
2,2024-01-01T02:00:00.000000,575.88,16.56
3,2024-01-01T03:00:00.000000,577.33,16.41
4,2024-01-01T04:00:00.000000,577.58,16.37


In [3]:
df["datetime"] = pd.to_datetime(
    df["datetime"],
    utc=True
)

df = (
    df
    .sort_values("datetime")
    .drop_duplicates(subset="datetime")
    .reset_index(drop=True)
)

print("Date range:")
print(df["datetime"].min(), "→", df["datetime"].max())

print("\nRows:", len(df))

Date range:
2024-01-01 00:00:00+00:00 → 2025-12-31 23:00:00+00:00

Rows: 17544


In [4]:
print("Missing values:")
print(df.isna().sum())

print("\nMissing percentage:")
print(
    (df.isna().mean() * 100)
    .round(3)
)

Missing values:
datetime                0
carbon_intensity        0
renewable_percentage    0
dtype: int64

Missing percentage:
datetime                0.0
carbon_intensity        0.0
renewable_percentage    0.0
dtype: float64


In [5]:
timestamp_diff = df["datetime"].diff().dt.total_seconds() / 3600

print("Unexpected time gaps:")
print(timestamp_diff.value_counts().sort_index().head(20))

Unexpected time gaps:
datetime
1.0    17543
Name: count, dtype: int64


In [6]:
expected_gap = 1

unexpected_gaps = timestamp_diff[
    timestamp_diff != expected_gap
].dropna()

print("\nNumber of unexpected gaps:", len(unexpected_gaps))

if len(unexpected_gaps) > 0:
    print("\nExamples:")
    print(unexpected_gaps.head(10))


Number of unexpected gaps: 0


In [7]:
print("Invalid CI:", (df["carbon_intensity"] <= 0).sum())

print(
    "Invalid RE:",
    (
        (df["renewable_percentage"] < 0) |
        (df["renewable_percentage"] > 100)
    ).sum()
)

Invalid CI: 0
Invalid RE: 0


In [8]:
df.loc[
    df["carbon_intensity"] <= 0,
    "carbon_intensity"
] = np.nan

df.loc[
    ~df["renewable_percentage"].between(0, 100),
    "renewable_percentage"
] = np.nan

In [9]:
for col in ["carbon_intensity", "renewable_percentage"]:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 3.5 * IQR
    upper = Q3 + 3.5 * IQR

    mask = (df[col] < lower) | (df[col] > upper)

    print(f"\n{col}")
    print("Lower:", lower)
    print("Upper:", upper)
    print("Potential outliers:", mask.sum())


carbon_intensity
Lower: 536.465
Upper: 632.545
Potential outliers: 1

renewable_percentage
Lower: 10.125000000000007
Upper: 21.404999999999994
Potential outliers: 181


In [10]:
for col in ["carbon_intensity", "renewable_percentage"]:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 3.5 * IQR
    upper = Q3 + 3.5 * IQR

    mask = (
        (df[col] < lower) |
        (df[col] > upper)
    )

    print(f"\n{col}")
    print(f"Lower: {lower:.3f}")
    print(f"Upper: {upper:.3f}")
    print(f"Potential outliers: {mask.sum()}")


carbon_intensity
Lower: 536.465
Upper: 632.545
Potential outliers: 1

renewable_percentage
Lower: 10.125
Upper: 21.405
Potential outliers: 181


In [11]:
fig = px.histogram(
    df,
    x="renewable_percentage",
    nbins=50,
    title="Distribution of Renewable Energy Percentage"
)

fig.show()

In [12]:
fig = px.line(
    df,
    x="datetime",
    y="renewable_percentage",
    title="Renewable Energy Percentage Over Time"
)

fig.show()

In [13]:
df["month"] = df["datetime"].dt.month

monthly_re = (
    df.groupby("month")["renewable_percentage"]
    .mean()
    .reset_index()
)

fig = px.bar(
    monthly_re,
    x="month",
    y="renewable_percentage",
    title="Monthly Mean Renewable Energy Percentage"
)

fig.show()

In [14]:
Q1 = df["renewable_percentage"].quantile(0.25)
Q3 = df["renewable_percentage"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 3.5 * IQR
upper = Q3 + 3.5 * IQR

outlier_mask = (
    (df["renewable_percentage"] < lower) |
    (df["renewable_percentage"] > upper)
)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=df["datetime"],
        y=df["renewable_percentage"],
        mode="lines",
        name="RE"
    )
)

fig.add_trace(
    go.Scatter(
        x=df.loc[outlier_mask, "datetime"],
        y=df.loc[outlier_mask, "renewable_percentage"],
        mode="markers",
        name="Potential statistical outliers",
        marker=dict(size=7)
    )
)

fig.update_layout(
    title="Potential Renewable Energy Outliers",
    xaxis_title="Datetime",
    yaxis_title="RE (%)"
)

fig.show()

In [15]:
col = "renewable_percentage"

Q1 = df[col].quantile(0.25)
Q3 = df[col].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 3.5 * IQR
upper = Q3 + 3.5 * IQR

outlier_mask = (
    (df[col] < lower) |
    (df[col] > upper)
)

outliers = df.loc[
    outlier_mask,
    ["datetime", col]
].copy()

In [16]:

print("=== OUTLIER SUMMARY ===")
print(f"Q1     : {Q1:.3f}")
print(f"Q3     : {Q3:.3f}")
print(f"IQR    : {IQR:.3f}")
print(f"Lower  : {lower:.3f}")
print(f"Upper  : {upper:.3f}")
print(f"Count  : {len(outliers)}")

=== OUTLIER SUMMARY ===
Q1     : 15.060
Q3     : 16.470
IQR    : 1.410
Lower  : 10.125
Upper  : 21.405
Count  : 181


In [17]:

print("\n=== OUTLIER VALUE DISTRIBUTION ===")
print(outliers[col].describe().round(3))




=== OUTLIER VALUE DISTRIBUTION ===
count    181.000
mean       9.539
std        0.437
min        8.300
25%        9.300
50%        9.640
75%        9.870
max       10.120
Name: renewable_percentage, dtype: float64


In [18]:
print("\n=== TEMPORAL RANGE ===")
print(f"First : {outliers['datetime'].min()}")
print(f"Last  : {outliers['datetime'].max()}")




=== TEMPORAL RANGE ===
First : 2025-07-01 14:00:00+00:00
Last  : 2025-07-31 20:00:00+00:00


In [19]:
print("\n=== FIRST 20 OUTLIERS ===")
print(outliers.head(20).to_string(index=False))




=== FIRST 20 OUTLIERS ===
                 datetime  renewable_percentage
2025-07-01 14:00:00+00:00                 10.12
2025-07-01 15:00:00+00:00                 10.12
2025-07-18 18:00:00+00:00                 10.11
2025-07-21 09:00:00+00:00                 10.04
2025-07-21 10:00:00+00:00                 10.09
2025-07-21 11:00:00+00:00                  9.91
2025-07-21 12:00:00+00:00                  9.93
2025-07-21 13:00:00+00:00                  9.82
2025-07-21 15:00:00+00:00                 10.05
2025-07-21 16:00:00+00:00                  9.96
2025-07-21 17:00:00+00:00                  9.53
2025-07-21 18:00:00+00:00                 10.03
2025-07-22 02:00:00+00:00                  9.96
2025-07-22 03:00:00+00:00                 10.03
2025-07-22 04:00:00+00:00                  9.86
2025-07-22 05:00:00+00:00                  9.93
2025-07-22 06:00:00+00:00                  9.86
2025-07-22 07:00:00+00:00                  9.71
2025-07-22 08:00:00+00:00                  9.75
2025-07-22 09

In [20]:
print("\n=== LAST 20 OUTLIERS ===")
print(outliers.tail(20).to_string(index=False))


=== LAST 20 OUTLIERS ===
                 datetime  renewable_percentage
2025-07-30 14:00:00+00:00                  9.64
2025-07-30 15:00:00+00:00                  9.57
2025-07-30 16:00:00+00:00                  9.61
2025-07-30 17:00:00+00:00                  9.71
2025-07-30 18:00:00+00:00                  9.94
2025-07-30 19:00:00+00:00                 10.10
2025-07-31 06:00:00+00:00                 10.01
2025-07-31 07:00:00+00:00                  9.90
2025-07-31 08:00:00+00:00                  9.34
2025-07-31 09:00:00+00:00                  9.34
2025-07-31 10:00:00+00:00                  9.13
2025-07-31 11:00:00+00:00                  9.13
2025-07-31 12:00:00+00:00                  9.32
2025-07-31 13:00:00+00:00                  9.39
2025-07-31 14:00:00+00:00                  9.54
2025-07-31 15:00:00+00:00                  9.50
2025-07-31 16:00:00+00:00                  9.59
2025-07-31 17:00:00+00:00                  9.87
2025-07-31 18:00:00+00:00                  9.99
2025-07-31 20:

In [21]:
outlier_times = outliers["datetime"].sort_values()

time_diff = outlier_times.diff()

print("\n=== TIME GAP BETWEEN OUTLIERS ===")
print(
    time_diff.value_counts().head(10)
)


=== TIME GAP BETWEEN OUTLIERS ===
datetime
0 days 01:00:00     168
0 days 02:00:00       3
0 days 11:00:00       2
17 days 03:00:00      1
2 days 15:00:00       1
0 days 08:00:00       1
0 days 13:00:00       1
0 days 05:00:00       1
0 days 16:00:00       1
0 days 14:00:00       1
Name: count, dtype: int64


In [22]:
outlier_times = outliers["datetime"].sort_values()

time_diff = outlier_times.diff()

print("\n=== TIME GAP BETWEEN OUTLIERS ===")
print(time_diff.value_counts().head(10))


=== TIME GAP BETWEEN OUTLIERS ===
datetime
0 days 01:00:00     168
0 days 02:00:00       3
0 days 11:00:00       2
17 days 03:00:00      1
2 days 15:00:00       1
0 days 08:00:00       1
0 days 13:00:00       1
0 days 05:00:00       1
0 days 16:00:00       1
0 days 14:00:00       1
Name: count, dtype: int64


In [23]:
print("\n=== MONTH DISTRIBUTION ===")

print(
    outliers["datetime"]
    .dt.to_period("M")
    .value_counts()
    .sort_index()
)


=== MONTH DISTRIBUTION ===
datetime
2025-07    181
Freq: M, Name: count, dtype: int64


/var/folders/9r/6wqjrtws4dg01ly8y_4l7yzw0000gn/T/ipykernel_28208/1544576992.py:5: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  .dt.to_period("M")


In [24]:
df_check = df[["datetime", "renewable_percentage"]].copy()

df_check["delta"] = df_check["renewable_percentage"].diff()
df_check["abs_delta"] = df_check["delta"].abs()

print("=== LARGEST HOURLY CHANGES ===")
print(
    df_check.nlargest(20, "abs_delta")
    .to_string(index=False)
)

=== LARGEST HOURLY CHANGES ===
                 datetime  renewable_percentage  delta  abs_delta
2025-08-01 00:00:00+00:00                 15.39   5.17       5.17
2025-07-01 00:00:00+00:00                 10.96  -5.06       5.06
2024-09-29 17:00:00+00:00                 16.89   2.00       2.00
2025-08-30 17:00:00+00:00                 16.28   1.86       1.86
2025-11-29 17:00:00+00:00                 16.14   1.69       1.69
2025-06-29 17:00:00+00:00                 15.81   1.62       1.62
2024-01-18 17:00:00+00:00                 14.75  -1.60       1.60
2025-03-30 17:00:00+00:00                 16.92   1.58       1.58
2025-01-25 17:00:00+00:00                 16.39   1.54       1.54
2024-12-31 17:00:00+00:00                 14.88  -1.53       1.53
2024-03-30 17:00:00+00:00                 16.46   1.51       1.51
2025-11-08 17:00:00+00:00                 17.01   1.48       1.48
2025-02-22 17:00:00+00:00                 16.10   1.47       1.47
2025-05-21 17:00:00+00:00                 13.

In [25]:
july_2025_mask = (
    (df["datetime"].dt.year == 2025) &
    (df["datetime"].dt.month == 7)
)

In [26]:
surrounding_mask = (
    (
        (df["datetime"].dt.year == 2025) &
        (df["datetime"].dt.month == 6)
    )
    |
    (
        (df["datetime"].dt.year == 2025) &
        (df["datetime"].dt.month == 8)
    )
)

In [27]:
july_mean_re = df.loc[
    july_2025_mask,
    "renewable_percentage"
].mean()

surrounding_mean_re = df.loc[
    surrounding_mask,
    "renewable_percentage"
].mean()

shift_re = surrounding_mean_re - july_mean_re

print(f"July 2025 RE mean    : {july_mean_re:.3f}%")
print(f"June-August baseline : {surrounding_mean_re:.3f}%")
print(f"Baseline difference  : {shift_re:.3f} percentage points")

July 2025 RE mean    : 10.635%
June-August baseline : 15.822%
Baseline difference  : 5.187 percentage points


In [28]:
if shift_re > 5:
    df.loc[
        july_2025_mask,
        "renewable_percentage"
    ] += shift_re

    print(
        f"July 2025 RE corrected by "
        f"+{shift_re:.3f} percentage points."
    )

July 2025 RE corrected by +5.187 percentage points.


In [29]:
col = "carbon_intensity"

Q1 = df[col].quantile(0.25)
Q3 = df[col].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 3.5 * IQR
upper = Q3 + 3.5 * IQR

ci_anomaly_mask = (
    (df[col] < lower) |
    (df[col] > upper)
)

print("=== CARBON INTENSITY ANOMALY ===")
print(f"Q1              : {Q1:.3f}")
print(f"Q3              : {Q3:.3f}")
print(f"IQR             : {IQR:.3f}")
print(f"Lower threshold : {lower:.3f}")
print(f"Upper threshold : {upper:.3f}")
print(f"Anomalies       : {ci_anomaly_mask.sum()}")

print("\nAnomalous observations:")

print(
    df.loc[
        ci_anomaly_mask,
        ["datetime", col]
    ].to_string(index=False)
)

df.loc[
    ci_anomaly_mask,
    col
] = np.nan

=== CARBON INTENSITY ANOMALY ===
Q1              : 578.500
Q3              : 590.510
IQR             : 12.010
Lower threshold : 536.465
Upper threshold : 632.545
Anomalies       : 1

Anomalous observations:
                 datetime  carbon_intensity
2025-07-25 08:00:00+00:00            632.65


In [30]:
july_mean_ci = df.loc[
    july_2025_mask,
    "carbon_intensity"
].mean()

surrounding_mean_ci = df.loc[
    surrounding_mask,
    "carbon_intensity"
].mean()

shift_ci = july_mean_ci - surrounding_mean_ci

print(f"July 2025 CI mean    : {july_mean_ci:.3f}")
print(f"June-August baseline : {surrounding_mean_ci:.3f}")
print(f"Baseline difference  : {shift_ci:.3f} gCO2eq/kWh")

July 2025 CI mean    : 614.400
June-August baseline : 584.733
Baseline difference  : 29.667 gCO2eq/kWh


In [31]:
if shift_ci > 15:
    df.loc[
        july_2025_mask,
        "carbon_intensity"
    ] -= shift_ci

    print(
        f"July 2025 CI corrected by "
        f"-{shift_ci:.3f} gCO2eq/kWh."
    )

July 2025 CI corrected by -29.667 gCO2eq/kWh.


In [32]:
print("Missing values before interpolation:")
print(
    df[
        ["carbon_intensity", "renewable_percentage"]
    ].isna().sum()
)

for col in [
    "carbon_intensity",
    "renewable_percentage"
]:
    df[col] = (
        df[col]
        .interpolate(method="linear")
        .ffill()
        .bfill()
    )

print("\nMissing values after interpolation:")
print(
    df[
        ["carbon_intensity", "renewable_percentage"]
    ].isna().sum()
)

Missing values before interpolation:
carbon_intensity        1
renewable_percentage    0
dtype: int64

Missing values after interpolation:
carbon_intensity        0
renewable_percentage    0
dtype: int64


In [33]:
print("=== FINAL DATA VALIDATION ===")

print("\nShape:")
print(df.shape)

print("\nMissing:")
print(
    df[
        [
            "carbon_intensity",
            "renewable_percentage"
        ]
    ].isna().sum()
)

print("\nCI range:")
print(
    df["carbon_intensity"].min(),
    "->",
    df["carbon_intensity"].max()
)

print("\nRE range:")
print(
    df["renewable_percentage"].min(),
    "->",
    df["renewable_percentage"].max()
)

print("\nDuplicate timestamps:")
print(
    df["datetime"].duplicated().sum()
)

print("\nTime gaps other than 1 hour:")
timestamp_diff = (
    df["datetime"]
    .diff()
    .dt.total_seconds()
    / 3600
)

print(
    timestamp_diff[
        timestamp_diff.notna() &
        (timestamp_diff != 1)
    ].value_counts()
)

=== FINAL DATA VALIDATION ===

Shape:
(17544, 4)

Missing:
carbon_intensity        0
renewable_percentage    0
dtype: int64

CI range:
558.03 -> 611.63

RE range:
12.67 -> 19.62

Duplicate timestamps:
0

Time gaps other than 1 hour:
Series([], Name: count, dtype: int64)


In [34]:
final_cols = [
    "datetime",
    "carbon_intensity",
    "renewable_percentage"
]

df_final = df[final_cols].copy()

df_final.to_csv(
    output_path,
    index=False
)

print(f"Saved  {output_path}")
print("Final shape:", df_final.shape)

Saved  ../data/processed/dataset_CI_RE_2024_2025_cleaned.csv
Final shape: (17544, 3)


In [35]:
import plotly.graph_objs as go

# Read in the data
df = pd.read_csv('../data/processed/dataset_CI_RE_2024_2025_cleaned.csv')

# Create a trace for the carbon intensity
trace1 = go.Scatter(x=df['datetime'], y=df['carbon_intensity'], name='Carbon Intensity', mode='lines+markers')

# Create a trace for the renewable percentage
trace2 = go.Scatter(x=df['datetime'], y=df['renewable_percentage'], name='Renewable Percentage', mode='lines+markers')

# Create a figure and add both traces to it
fig = go.Figure()
fig.add_trace(trace1)
fig.add_trace(trace2)

# Set the title of the graph
fig.update_layout(title='Carbon Intensity vs Renewable Percentage')

# Display the graph
fig.show()